# Querying the Arretine chronology graph

The same eleven queries as the published query page, as a plain notebook. It runs locally against the Turtle files in `output/`, so it needs only rdflib and pandas — no Quarto, no Pyodide, no endpoint. Page, notebook and the quarto-live variant are all generated from `queries.yaml`, and the figures are the same files under `py/viz/`, so none of the three can drift from the others.
This is not the same thing as `clades_variana_temporal.ipynb`, which is an argument about one event and was written independently. Its queries could be folded in here later; until then the two overlap in what they ask and are maintained separately.

## Setup

In [ ]:
import json
import logging
from html import escape
from pathlib import Path

import pandas as pd
from rdflib import Graph

# This graph dates events BC, so it carries xsd:gYear literals with negative
# years. rdflib tries to map each onto a datetime.date, whose minimum year is
# 1, fails, and logs a traceback per literal. Nothing is wrong - the queries
# read the lexical form via STR() - but the tracebacks look alarming.
logging.getLogger("rdflib.term").setLevel(logging.ERROR)

PREFIXES = """
PREFIX ae:        <http://leiza-scit.github.io/CAA2026-alligator/>
PREFIX aecol:     <http://leiza-scit.github.io/CAA2026-alligator/collections/>
PREFIX crm:       <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX aeont:     <http://leiza-scit.github.io/CAA2026-alligator/ontology#>
PREFIX fsl:       <http://fuzzy-sl.squirrel.link/ontology/>
PREFIX geosparql: <http://www.opengis.net/ont/geosparql#>
PREFIX lado:      <http://archaeology.link/ontology#>
PREFIX rdfs:      <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos:      <http://www.w3.org/2004/02/skos/core#>
PREFIX time:      <http://www.w3.org/2006/time#>
PREFIX xsd:       <http://www.w3.org/2001/XMLSchema#>
"""


def _load(name):
    """Parse a Turtle file, whether it sits beside this notebook or in output/."""
    for candidate in (Path(name), Path("..") / "output" / name):
        if candidate.exists():
            return Graph().parse(candidate, format="turtle")
    raise FileNotFoundError(
        f"{name} found neither here nor in ../output/ - "
        f"run python py/main.py first.")


g = _load('arretine_sites_minigraph.ttl')
print(f"{len(g):,} triples from arretine_sites_minigraph.ttl")
services = _load('arretine_services.ttl')
print(f"  + {len(services):,} triples from arretine_services.ttl")


def show(graph, query):
    """Run a query and return its rows as a list of dicts of strings.

    Strings, deliberately: it is what the figures expect, and it keeps this
    notebook and the browser variants reading the same values rather than one
    of them quietly getting a typed object the others never see.
    """
    result = graph.query(PREFIXES + query)
    columns = [str(v) for v in result.vars]
    return [{c: (None if row[i] is None else str(row[i]))
             for i, c in enumerate(columns)} for row in result]


def table(rows):
    """A query result as a DataFrame, for display."""
    return pd.DataFrame(rows) if rows else pd.DataFrame()


# Each query's rows are kept here under its id, so a figure cell can be re-run
# on its own without the rows underneath it having silently changed.
results = {}


class Frame:
    """A self-contained HTML document, shown inline as a figure.

    The document goes into an iframe rather than straight into the output. That
    keeps the figure's CSS and element ids away from the notebook, and it means
    the scripts inside actually run - a <script> injected into a notebook's
    output area does not.
    """

    def __init__(self, html, height=520):
        self.html = html
        self.height = height

    def _repr_html_(self):
        # Only these two need escaping inside a double-quoted attribute, and
        # the ampersand has to go first or it would escape the escapes.
        doc = self.html.replace("&", "&amp;").replace('"', "&quot;")
        return (f'<iframe srcdoc="{doc}" loading="lazy"'
                f' sandbox="allow-scripts allow-popups'
                f' allow-popups-to-escape-sandbox"'
                f' style="width:100%;height:{self.height}px;border:0">'
                f'</iframe>')


# Shared helpers for the figures in py/viz/, inlined into the notebook's setup
# cell (queries.yaml -> qmd.viz_prelude). Nothing here is generic: it is the
# vocabulary this particular graph is discussed in.
#
# This file is *not* imported. It is pasted into a Pyodide cell, so it may only
# use the standard library and must not assume a working directory.

# The Allen relation palette from alligator_to_clean_rdf.py, so a relation is
# the same colour in the browser figures and in the figures of the paper.
ALLEN_COLOUR = {
    "before":       "#4a90d9",   # blue
    "after":        "#2c5f8a",   # dark blue
    "meets":        "#7ab3e0",   # light blue
    "metBy":        "#5a9fc5",   # mid blue
    "overlaps":     "#f0a500",   # orange
    "overlappedBy": "#c97d00",   # dark orange
    "contains":     "#d94a4a",   # red
    "during":       "#a03030",   # dark red
    "starts":       "#e07070",   # light red
    "startedBy":    "#c05050",   # mid red
    "finishes":     "#e09090",   # pink-red
    "finishedBy":   "#b04060",   # rose
    "equals":       "#4caf50",   # green
}

# Allen's own ordering, from wholly earlier to wholly later. Sorting the bar
# chart by frequency would hide that the occupied relations form one block.
ALLEN_ORDER = [
    "before", "meets", "overlaps", "finishedBy", "contains", "starts",
    "equals", "startedBy", "during", "finishes", "overlappedBy", "metBy",
    "after",
]

# Relations that mean "contemporary with, or later than" the reference event,
# copied from the companion notebook so both give the same answer. 'meets' is
# deliberately included: an interval ending exactly where the event begins was
# still in use when it began.
#
# 'overlaps' is the one omission, and it is worth knowing about. Against a
# single-year event it would be unreachable, but the graph dates the Clades
# Variana AD 8 to 9, so a findspot ending in AD 8 would fall into it — and
# would then be dropped, although it was in use when the event opened. No
# findspot in this corpus does, so the two readings agree here; whether they
# should agree in general is a question for Allard.
CONTEMPORARY_OR_LATER = {
    "after", "metBy", "equals", "during", "starts", "startedBy", "finishes",
    "finishedBy", "overlappedBy", "contains", "meets",
}

# One colour per chronological horizon, cold (early) to warm (late), so the
# map can be read as a sequence without consulting the legend.
HORIZON_COLOUR = {
    "1": "#0d4a70",
    "2": "#2e86ab",
    "3": "#7fb800",
    "4": "#f0a500",
    "5": "#c1440e",
}

# The reference event, for the two temporal figures. This is the year the
# timeline marks, i.e. the defeat itself; the Allen relations in the query are
# computed against the full interval the graph records, AD 8 to 9.
EVENT_LABEL = "Clades Variana"
EVENT_YEAR = 9
EVENT_COLOUR = "#993c1d"


# The service-type palette, taken from py/events_timeline_by_service.py by
# sampling the same colormaps at the same points (Reds 0.35-0.90 over the six
# Service I types, Greens 0.45-0.88 over the two Service II types). Written out
# here because Pyodide has no matplotlib: a type must be the same colour in the
# browser as in the printed figure, and the only way to guarantee that without
# matplotlib is to carry the values.
SERVICE_COLOUR = {
    "Schrägrandteller":  "#1f77b4",
    "Service Ia Tasse":  "#fc9b7c",
    "Service Ia Teller": "#fb7757",
    "Service Ib Tasse":  "#f4503a",
    "Service Ib Teller": "#de2b25",
    "Service Ic Tasse":  "#be151a",
    "Service Ic Teller": "#980c13",
    "Service II Tasse":  "#86cc85",
    "Service II Teller": "#006b2b",
}

# Fill for a heatmap cell whose value follows from the rank assignment alone
# rather than from the sherds — see the quality figure. Same grey as the
# printed version uses for those cells.
STRUCTURAL_GREY = "#eeeeee"

# Matplotlib's RdYlGn and YlOrRd, sampled at eleven stops. The printed figures
# use them directly; Pyodide has no matplotlib, so the stops travel with the
# code and ramp() interpolates between them. Same colormap, same values, same
# colours in the browser as on the page.
RDYLGN = ["#a50026", "#d62f27", "#f46d43", "#fdad60", "#fee08b", "#feffbe",
          "#d9ef8b", "#a5d86a", "#66bd63", "#199750", "#006837"]
YLORRD = ["#ffffcc", "#fff1a9", "#fee187", "#feca66", "#feab49", "#fd8c3c",
          "#fc5b2e", "#ed2e21", "#d41020", "#b00026", "#800026"]


def ramp(stops, t):
    """Colour at position t in [0, 1] along a list of hex stops."""
    t = min(max(t, 0.0), 1.0)
    span = t * (len(stops) - 1)
    i = min(int(span), len(stops) - 2)
    f = span - i
    a = [int(stops[i][k:k + 2], 16) for k in (1, 3, 5)]
    b = [int(stops[i + 1][k:k + 2], 16) for k in (1, 3, 5)]
    return "#%02x%02x%02x" % tuple(round(a[k] + (b[k] - a[k]) * f)
                                   for k in range(3))


def ink_on(hex_colour):
    """Black or white text, whichever stays legible on the given fill."""
    r, g, b = (int(hex_colour[k:k + 2], 16) for k in (1, 3, 5))
    return "#1a1a1a" if (0.299 * r + 0.587 * g + 0.114 * b) > 150 else "#ffffff"


def colourbar(stops, lo, hi, width=190, height=10, fmt="{:.1f}"):
    """A horizontal colourbar as pure SVG.

    Drawn as thin segments rather than a gradient element so the figure stays
    a plain vector that survives being saved or printed — the same reason the
    printed figures avoid a rasterised colorbar.
    """
    steps = 60
    bars = "".join(
        f'<rect x="{i * width / steps:.2f}" y="0" '
        f'width="{width / steps + 0.6:.2f}" height="{height}" '
        f'fill="{ramp(stops, i / (steps - 1))}"/>' for i in range(steps))
    ticks = "".join(
        f'<text x="{f * width:.1f}" y="{height + 11}" font-size="9" '
        f'fill="#888" text-anchor="{a}">{fmt.format(lo + f * (hi - lo))}</text>'
        for f, a in ((0.0, "start"), (0.5, "middle"), (1.0, "end")))
    return (f'<svg width="{width}" height="{height + 15}" '
            f'xmlns="http://www.w3.org/2000/svg">{bars}{ticks}</svg>')


def rgzm(n, sum_rank, sum_rank_sq):
    """RGZM within-group variance and quality from the sums a query can return.

    SPARQL has no EXP or SQRT — rdflib's does not even parse them — so the query
    returns the three sums and the last step happens here:

        x̄  = Σ(rank·count) / N
        s  = sqrt( (Σ(rank²·count) − N·x̄²) / (N − 1) )    (STDDEV_SAMP)
        q  = exp(−s/|x̄|)                                  (quality, in (0, 1])

    Every sherd is one observation valued by the rank of its sub-type, which is
    what makes the sums above sufficient. q → 1 means the group's material sits
    on one rank; q → 0 means it is spread across the group's sequence.
    """
    import math

    if n < 2:
        return float("nan"), float("nan")
    mean = sum_rank / n
    variance = (sum_rank_sq - n * mean * mean) / (n - 1)
    s = math.sqrt(max(variance, 0.0))
    if mean == 0:
        return s, float("nan")
    return s, math.exp(-(s / abs(mean)))


def year(value):
    """Astronomical year number to a reading label: -9 -> '9 BC', 9 -> 'AD 9'.

    The graph stores xsd:gYear in the proleptic Gregorian calendar, which has a
    year zero; historical year numbering does not. Within this corpus nothing
    is dated to year 0, so the shift can be ignored and the conversion is the
    plain sign flip below.
    """
    v = int(value)
    return f"{-v} BC" if v < 0 else f"AD {v}"


# Service-type colours, sampled from the same matplotlib colormaps and the same
# ranges as build_service_colours() in events_timeline_by_service.py, so a type
# is the same colour in the browser as in the figures of the paper.
# Schrägrandteller is a fixed blue; Service I runs light to dark red over its
# six sub-types, Service II light to dark green over its two.
SERVICE_COLOUR = {
    "Schraegrandteller": "#1f77b4",
    "ServiceIa_Tasse":   "#fc9b7c",
    "ServiceIa_Teller":  "#fb7757",
    "ServiceIb_Tasse":   "#f4503a",
    "ServiceIb_Teller":  "#de2b25",
    "ServiceIc_Tasse":   "#be151a",
    "ServiceIc_Teller":  "#980c13",
    "ServiceII_Tasse":   "#86cc85",
    "ServiceII_Teller":  "#006b2b",
}

# Rows of the horizon figures run latest at the top, the timeline convention.
HORIZON_DISPLAY_ORDER = ["5", "4", "3", "2", "1"]

## Notes

**Two groupings, one graph.** A `lado:PeriodCluster` groups findspots that the
seriation gave exactly the same start and end year. A
`lado:ChronologicalHorizon` is the phase a findspot was *assigned* to.
Horizons 3–5 coincide with a single cluster each, but horizon 2 absorbs three
clusters and horizon 1 two, so a horizon's interval is the *envelope* of its
members and can be wider than any findspot in it.

**Astronomical year numbering.** Dates are `xsd:gYear` literals in the
proleptic Gregorian calendar, which has a year zero: `"-0008"` is 9 BC, not
8 BC. rdflib does not compare `xsd:gYear` literals the way you would expect
either — cast them first, as the queries below do with
`xsd:integer(STR(?year))`.

**Fixed versus estimated boundaries.** `lado:startfixed` and `lado:endfixed`
mark boundaries resting on a historically dated event rather than on
seriation alone. Only seven findspots carry such an anchor, and the whole
chronology hangs on them.

**Figures.** Three of the queries below draw something rather than print a
table. They are the same figures as in `notebook/clades_variana_temporal
.ipynb`, rebuilt from the query result rather than from a pandas frame, so
the browser notebook and the Jupyter one show the same thing. The map
fetches tiles from openstreetmap.org; everything else is computed from the
Turtle file alone.

## 1 · The five chronological horizons

The starting point for everything else: each horizon with its interval and how many findspots it holds. Horizon 1 is the earliest, horizon 5 the latest. The interval is the envelope of the horizon's findspots, from the earliest start to the latest end.

In [ ]:
rows = show(g, r"""
SELECT ?horizon ?label ?begin ?end (COUNT(?site) AS ?findspots)
WHERE {
  ?h a lado:ChronologicalHorizon ;
     skos:notation                     ?horizon ;
     rdfs:label                        ?label ;
     time:hasBeginning/time:inXSDgYear ?begin ;
     time:hasEnd/time:inXSDgYear       ?end ;
     lado:hasHorizonMember             ?site .
  FILTER(langMatches(LANG(?label), "en"))
}
GROUP BY ?horizon ?label ?begin ?end
ORDER BY ?horizon""")
results['horizons-overview'] = rows

print(f"{len(rows)} rows")
table(rows)

### The horizons as intervals

The same figure as `output/horizon_timeline_en.svg`, drawn from the query above. Colour is the number of findspots. Horizons 1 and 2 both open in 15 BC and horizon 2 runs to AD 13, so the sequence is not a simple ladder — an envelope can swallow the one after it.

In [ ]:
rows = results['horizons-overview']

# Figure: the five chronological horizons as intervals, coloured by how many
# findspots each holds. The browser counterpart of output/horizon_timeline_en.svg
# and drawn to match it: latest horizon at the top, bar labelled inside where it
# fits and outside where it does not, YlOrRd sampled the same way the printed
# version samples it, colourbar from 1 to the largest horizon.
#
# Runs in a Pyodide cell with `rows` (the query result) and the helpers from
# py/viz/_prelude.py in scope. It must end in a Frame.

horizons = [{
    "horizon": r["horizon"],
    "label": r["label"],
    "begin": int(str(r["begin"]).lstrip("+")),
    "end": int(str(r["end"]).lstrip("+")),
    "findspots": int(r["findspots"]),
} for r in rows]

most = max(h["findspots"] for h in horizons)

# The printed figure samples YlOrRd over 0.3 to 1.0 rather than the whole
# range: the pale end of the map is nearly white and a bar in it disappears.
for h in horizons:
    h["colour"] = ramp(YLORRD, 0.3 + 0.7 * h["findspots"] / most)
    h["ink"] = ink_on(h["colour"])

span_min = min(h["begin"] for h in horizons)
span_max = max(h["end"] for h in horizons)

# The right margin has to hold the longest string that can end up outside a
# bar, or it is silently clipped — which is what happened to horizon 5. There
# is no text metric available here, so width is estimated from the character
# count; the constant is deliberately generous.
longest = max(len(f'{h["label"]} · {h["findspots"]} findspots')
              for h in horizons)

payload = json.dumps({
    "horizons": sorted(horizons, key=lambda h: h["horizon"]),
    "min": span_min - 3, "max": span_max + 5,
    "rightMargin": round(longest * 6.6) + 24,
})

bar = colourbar(YLORRD[3:], 1, most, width=200, fmt="{:.0f}")

style = """
  body{margin:0;font-family:sans-serif;padding:6px 4px 4px}
  #chart{overflow-x:auto}
  #foot{display:flex;align-items:flex-end;gap:10px;font-size:11px;color:#777;
    padding:.5rem 0 0}
"""

script = """
(function () {
  var C = JSON.parse(document.getElementById("payload").textContent);
  var ce = document.getElementById("chart");

  function yr(v) { return v < 0 ? (-v) + " BC" : "AD " + v; }

  var RH = 46, RG = 14, MT = 16, MB = 34, LW = 8, CW = 660,
      RW = C.rightMargin;
  var d = C.horizons.slice().reverse();          // latest at the top
  var W = LW + CW + RW, H = MT + d.length * (RH + RG) + MB;
  function px(y) { return LW + (y - C.min) / (C.max - C.min) * CW; }

  var o = '<svg xmlns="http://www.w3.org/2000/svg" width="' + W + '"'
        + ' height="' + H + '" style="font-family:sans-serif">';
  o += '<rect x="' + LW + '" y="' + MT + '" width="' + CW + '"'
     + ' height="' + (H - MT - MB) + '" fill="#f7f7f7"/>';

  var step = 5, t;
  for (t = Math.ceil(C.min / step) * step; t <= C.max; t += step) {
    var p = px(t);
    o += '<line x1="' + p + '" y1="' + MT + '" x2="' + p + '"'
       + ' y2="' + (H - MB) + '" stroke="#e3e3e3" stroke-width="1"/>';
    o += '<text x="' + p + '" y="' + (H - MB + 16) + '" font-size="10"'
       + ' fill="#666" text-anchor="end" transform="rotate(-35 ' + p + ','
       + (H - MB + 16) + ')">' + yr(t) + "</text>";
  }

  d.forEach(function (h, i) {
    var y = MT + i * (RH + RG) + RG / 2;
    var x1 = px(h.begin), w = Math.max(px(h.end) - x1, 2);
    o += '<rect x="' + x1 + '" y="' + y + '" width="' + w + '"'
       + ' height="' + RH + '" fill="' + h.colour + '">'
       + "<title>" + h.label + " \\u00b7 " + h.findspots
       + " findspots</title></rect>";
    // The count goes on every bar. Printing it only when the label fits
    // inside meant four of the five horizons silently lost it.
    //
    // "Fits" is measured against the label rather than against a fixed bar
    // width: at 210px horizon 5 was pushed outside although its bar is wide
    // enough to hold the text, which is where the printed figure puts it.
    var count = h.findspots + " findspots";
    var inside = w > h.label.length * 6.9 + 18;
    if (inside) {
      o += '<text x="' + (x1 + w / 2) + '" y="' + (y + RH / 2 + 4) + '"'
         + ' font-size="12.5" font-weight="600" text-anchor="middle"'
         + ' fill="' + h.ink + '">' + h.label + "</text>";
      o += '<text x="' + (x1 + w + 8) + '" y="' + (y + RH / 2 + 4) + '"'
         + ' font-size="11.5" fill="#555">' + count + "</text>";
    } else {
      o += '<text x="' + (x1 + w + 8) + '" y="' + (y + RH / 2 + 4) + '"'
         + ' font-size="12.5" font-weight="600" fill="#333">' + h.label
         + '<tspan font-weight="400" fill="#666"> \u00b7 ' + count
         + "</tspan></text>";
    }
  });

  o += "</svg>";
  ce.innerHTML = o;
})();
"""

timeline = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><style>{style}</style></head>
<body>
<div id="chart"></div>
<div id="foot">
  <span>Number of findspots</span>{bar}
  <span>Bars span the envelope of the horizon's findspots, from the earliest
  start to the latest end &mdash; so two horizons can overlap.</span>
</div>
<script id="payload" type="application/json">{payload}</script>
<script>{script}</script>
</body></html>"""

Frame(timeline, height=len(horizons) * 60 + 130)

## 2 · How the horizons relate to one another

The graph carries all twenty pairwise Allen relations between the horizons, so the sequence can be queried rather than read off a figure. Horizon 1 *meets* horizon 3 — one ends exactly where the other begins — while horizon 2 *contains* horizon 3 entirely.

In [ ]:
rows = show(g, r"""
SELECT ?horizonA ?beginA ?endA ?relation ?horizonB
WHERE {
  ?a a lado:ChronologicalHorizon ;
     skos:notation                     ?horizonA ;
     time:hasBeginning/time:inXSDgYear ?beginA ;
     time:hasEnd/time:inXSDgYear       ?endA ;
     ?rel                              ?b .
  ?b a lado:ChronologicalHorizon ; skos:notation ?horizonB .
  FILTER(STRSTARTS(STR(?rel), STR(time:)))
  BIND(REPLACE(STR(?rel), "^.*#interval", "") AS ?relation)
}
ORDER BY ?horizonA ?horizonB""")
results['horizon-succession'] = rows

print(f"{len(rows)} rows")
table(rows)

### The succession as a matrix

The same figure as `output/horizon_allen_matrix_en.svg`. Read a cell as “row relation column”. Only the blue family means one horizon is wholly done before the next begins; the red cells are the pairs whose intervals contain one another, which is where the sequence stops being a line.

In [ ]:
rows = results['horizon-succession']

# Figure: Allen's interval relation between every ordered pair of horizons, as a
# matrix. The browser counterpart of output/horizon_allen_matrix_en.svg, and
# drawn to match it: the same relation palette, the same three families in the
# legend, the diagonal greyed out, axis labels carrying each horizon's interval.
#
# The palette is not re-invented here — ALLEN_COLOUR in py/viz/_prelude.py is
# copied from alligator_to_clean_rdf.py, so a relation is the same colour in
# this matrix as in the printed one.
#
# Runs in a Pyodide cell with `rows` (the query result) and the helpers from
# py/viz/_prelude.py in scope. It must end in a Frame.

# The query names the relation as OWL-Time does, in CamelCase; the palette and
# the printed labels use Allen's own spelling.
SHORT = {
    "Before": ("before", "before"), "After": ("after", "after"),
    "Meets": ("meets", "meets"), "MetBy": ("metBy", "met-by"),
    "Overlaps": ("overlaps", "overlaps"),
    "OverlappedBy": ("overlappedBy", "ovlp-by"),
    "Contains": ("contains", "contains"), "During": ("during", "during"),
    "Starts": ("starts", "starts"), "StartedBy": ("startedBy", "started-by"),
    "Finishes": ("finishes", "finishes"),
    "FinishedBy": ("finishedBy", "finished-by"),
    "Equals": ("equals", "equals"),
}

FAMILIES = [
    ("Sequential (before / after / meets / met-by)", "#4a90d9",
     {"before", "after", "meets", "metBy"}),
    ("Overlapping (overlaps / overlapped-by)", "#f0a500",
     {"overlaps", "overlappedBy"}),
    ("Containing (contains / during / starts / finishes \u2026)", "#d94a4a",
     {"contains", "during", "starts", "startedBy", "finishes", "finishedBy"}),
    ("Equal", "#4caf50", {"equals"}),
]


def year(value):                                   # local, tolerates gYear text
    v = int(str(value).lstrip("+"))
    return f"{-v}BC" if v < 0 else f"AD{v}"


spans, cells = {}, {}
for r in rows:
    a, b = r["horizonA"], r["horizonB"]
    key, label = SHORT.get(r["relation"], (r["relation"], r["relation"]))
    cells[(a, b)] = {"key": key, "label": label,
                     "colour": ALLEN_COLOUR.get(key, "#888888")}
    if "beginA" in r and r["beginA"] is not None:
        spans[a] = f'{year(r["beginA"])}\u2013{year(r["endA"])}'

axis = sorted({h for pair in cells for h in pair})
head = "".join(
    f'<th><span>H{escape(h)}</span>'
    f'<em>{escape(spans.get(h, ""))}</em></th>' for h in axis)

body = ""
for a in axis:
    body += (f'<tr><th class="side"><span>H{escape(a)}</span>'
             f'<em>{escape(spans.get(a, ""))}</em></th>')
    for b in axis:
        if a == b:
            body += '<td class="same" title="the same horizon">&mdash;</td>'
            continue
        cell = cells.get((a, b))
        if cell is None:
            body += '<td class="same">&nbsp;</td>'
            continue
        tip = (f'H{a} ({spans.get(a, "?")}) {cell["label"]} '
               f'H{b} ({spans.get(b, "?")})')
        body += (f'<td style="background:{cell["colour"]};'
                 f'color:{ink_on(cell["colour"])}" title="{escape(tip)}">'
                 f'{escape(cell["label"])}</td>')
    body += "</tr>"

legend = "".join(
    f'<span><b style="background:{colour}"></b>{escape(name)}</span>'
    for name, colour, _members in FAMILIES
    if any(c["key"] in _members for c in cells.values())
) + '<span><b style="background:#e0e0e0"></b>Same horizon</span>'

style = """
  body{margin:0;font-family:sans-serif;padding:6px 4px 4px;color:#333}
  .axis{font-size:11px;color:#888;text-align:center;margin:0 0 6px}
  table{border-collapse:separate;border-spacing:3px;margin:0 auto}
  th{font-weight:600;font-size:11.5px;color:#444;padding:2px 4px}
  th span{display:block}
  th em{display:block;font-style:normal;font-size:9.5px;color:#999;
    font-weight:400}
  th.side{text-align:right}
  td{width:104px;height:52px;text-align:center;border-radius:4px;
    font-size:11.5px;font-weight:600;cursor:help}
  td.same{background:#e0e0e0;color:#aaa;font-weight:400;cursor:default}
  #leg{display:flex;gap:14px;flex-wrap:wrap;justify-content:center;
    font-size:11px;color:#666;padding:.7rem 0 0}
  #leg span{display:flex;align-items:center;gap:5px}
  #leg b{width:11px;height:11px;border-radius:2px;display:inline-block}
"""

matrix = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><style>{style}</style></head>
<body>
<p class="axis">row = horizon A &nbsp;&middot;&nbsp; column = horizon B
  &nbsp;&middot;&nbsp; read as &ldquo;A <em>relation</em> B&rdquo;</p>
<table><thead><tr><th></th>{head}</tr></thead><tbody>{body}</tbody></table>
<div id="leg">{legend}</div>
</body></html>"""

Frame(matrix, height=len(axis) * 58 + 150)

## 3 · Which findspots anchor the chronology?

Alligator marks a boundary as fixed when it rests on a historically dated event rather than on seriation alone. Only seven findspots carry such an anchor — and every horizon has at least one, which is what makes the sequence defensible.

In [ ]:
rows = show(g, r"""
SELECT ?horizon ?findspot ?startFixed ?endFixed ?begin ?end
WHERE {
  ?h a lado:ChronologicalHorizon ;
     skos:notation         ?horizon ;
     lado:hasHorizonMember ?site .

  ?site rdfs:label                        ?findspot ;
        lado:startfixed                   ?startFixed ;
        lado:endfixed                     ?endFixed ;
        time:hasBeginning/time:inXSDgYear ?begin ;
        time:hasEnd/time:inXSDgYear       ?end .

  FILTER(?startFixed || ?endFixed)
}
ORDER BY ?horizon ?findspot""")

print(f"{len(rows)} rows")
table(rows)

## 4 · Findspots that do not fill their horizon

Where every findspot carries the same dating, the horizon envelope describes the horizon exactly. Where members differ, the envelope is wider than any single findspot. This query returns only the findspots narrower than their horizon — none of them are in horizons 1 to 3, which is the quickest evidence that those three are internally homogeneous.

In [ ]:
rows = show(g, r"""
SELECT ?horizon ?findspot ?siteBegin ?siteEnd ?horizonBegin ?horizonEnd
WHERE {
  ?h a lado:ChronologicalHorizon ;
     skos:notation                     ?horizon ;
     time:hasBeginning/time:inXSDgYear ?horizonBegin ;
     time:hasEnd/time:inXSDgYear       ?horizonEnd ;
     lado:hasHorizonMember             ?site .

  ?site rdfs:label                        ?findspot ;
        time:hasBeginning/time:inXSDgYear ?siteBegin ;
        time:hasEnd/time:inXSDgYear       ?siteEnd .

  FILTER(xsd:integer(STR(?siteBegin)) != xsd:integer(STR(?horizonBegin))
      || xsd:integer(STR(?siteEnd))   != xsd:integer(STR(?horizonEnd)))
}
ORDER BY ?horizon ?findspot""")

print(f"{len(rows)} rows")
table(rows)

## 5 · Which seriation clusters does each horizon absorb?

Both groupings live in the same graph, so one query can join them. A period cluster is computed — findspots the seriation gave an identical start and end year; a horizon is interpretive. Horizons 3 to 5 rest on a single cluster each, horizons 1 and 2 merge several, which is exactly why their envelopes are wider than their members.

In [ ]:
rows = show(g, r"""
SELECT ?horizon ?cluster (COUNT(?site) AS ?findspots)
WHERE {
  ?h a lado:ChronologicalHorizon ;
     skos:notation         ?horizon ;
     lado:hasHorizonMember ?site .

  ?c a lado:PeriodCluster ;
     rdfs:label            ?cluster ;
     lado:hasClusterMember ?site .
}
GROUP BY ?horizon ?cluster
ORDER BY ?horizon ?cluster""")

print(f"{len(rows)} rows")
table(rows)

## 6 · Which findspots were still in use in AD 9?

The question the companion notebook works through in three different ways. Here it is resolved on the horizon level and in plain SPARQL: a findspot qualifies when its own interval covers AD 9, the year of the Clades Variana. Note the cast — rdflib will not compare xsd:gYear literals directly.

In [ ]:
rows = show(g, r"""
SELECT ?horizon ?findspot ?begin ?end
WHERE {
  ?h a lado:ChronologicalHorizon ;
     skos:notation         ?horizon ;
     lado:hasHorizonMember ?site .

  ?site rdfs:label                        ?findspot ;
        time:hasBeginning/time:inXSDgYear ?begin ;
        time:hasEnd/time:inXSDgYear       ?end .

  FILTER(xsd:integer(STR(?begin)) <= 9 && xsd:integer(STR(?end)) >= 9)
}
ORDER BY ?horizon ?findspot""")

print(f"{len(rows)} rows")
table(rows)

## 7 · Every findspot's Allen relation to the Clades Variana

The companion notebook computes Allen's thirteen interval relations in pandas. They can be derived in the query instead, with a chain of nested IFs testing the cases in Allen's order. Note that the event's interval is read from the graph rather than written into the query: the Clades Variana is recorded as AD 8 to 9, not as the single year AD 9, and hard-coding the year would quietly give different answers. The 44 findspots fall into five of the thirteen relations, and both figures below are drawn from these rows.

In [ ]:
rows = show(g, r"""
SELECT ?site ?findspot ?begin ?end ?relation
WHERE {
  ae:event_Clades_Variana
      time:hasBeginning/time:inXSDgYear ?eventBeginYear ;
      time:hasEnd/time:inXSDgYear       ?eventEndYear .

  ?s a fsl:Site , time:Interval ;
     rdfs:label                        ?findspot ;
     time:hasBeginning/time:inXSDgYear ?beginYear ;
     time:hasEnd/time:inXSDgYear       ?endYear .

  BIND(xsd:integer(STR(?beginYear))      AS ?begin)
  BIND(xsd:integer(STR(?endYear))        AS ?end)
  BIND(xsd:integer(STR(?eventBeginYear)) AS ?eventBegin)
  BIND(xsd:integer(STR(?eventEndYear))   AS ?eventEnd)
  BIND(REPLACE(STR(?s), "^.*/", "")      AS ?site)

  BIND(IF(?end   <  ?eventBegin, "before",
       IF(?begin >  ?eventEnd,   "after",
       IF(?end   =  ?eventBegin, "meets",
       IF(?begin =  ?eventEnd,   "metBy",
       IF(?begin =  ?eventBegin && ?end =  ?eventEnd, "equals",
       IF(?begin <  ?eventBegin && ?end >  ?eventEnd, "contains",
       IF(?begin >  ?eventBegin && ?end <  ?eventEnd, "during",
       IF(?begin =  ?eventBegin && ?end <  ?eventEnd, "starts",
       IF(?begin =  ?eventBegin && ?end >  ?eventEnd, "startedBy",
       IF(?begin <  ?eventBegin && ?end =  ?eventEnd, "finishedBy",
       IF(?begin >  ?eventBegin && ?end =  ?eventEnd, "finishes",
       IF(?begin <  ?eventBegin && ?end <  ?eventEnd, "overlaps",
       IF(?begin >  ?eventBegin && ?end >  ?eventEnd, "overlappedBy",
                                                      "unknown")))))))))))))
       AS ?relation)
}
ORDER BY ?begin ?findspot""")
results['allen-relations-to-clades-variana'] = rows

print(f"{len(rows)} rows")
table(rows)

### Distribution across the thirteen relations

Coloured bars are the relations that count as contemporary with or later than the event; grey is *before*. Only five of the thirteen are occupied, and that is worth pausing on: the event's interval, AD 8 to 9, falls exactly on two of the boundaries the seriation produced, so most findspots either end precisely where it ends (*finishedBy*, 21) or begin precisely where it begins (*startedBy*, 10). The relation is reporting a coincidence of boundaries, not a smooth distribution.

In [ ]:
rows = results['allen-relations-to-clades-variana']

# Figure: how the 44 findspots distribute across Allen's thirteen relations to
# the Clades Variana. Ported from section 10 of notebook/clades_variana_temporal
# .ipynb, which built the same chart from a relation computed in pandas; here
# the relation arrives from the query above, computed in SPARQL.
#
# Runs in a Pyodide cell with `rows` (the query result) and the helpers from
# py/viz/_prelude.py in scope. It must end in a Frame.

counts = {rel: 0 for rel in ALLEN_ORDER}
for r in rows:
    counts[r["relation"]] = counts.get(r["relation"], 0) + 1

labels = ALLEN_ORDER
values = [counts[rel] for rel in labels]
colours = [ALLEN_COLOUR[rel] if rel in CONTEMPORARY_OR_LATER else "#b4b2a9"
           for rel in labels]

occupied = sum(1 for v in values if v)
qualifying = sum(v for rel, v in zip(labels, values)
                 if rel in CONTEMPORARY_OR_LATER)

chart = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js">
</script>
</head>
<body style="margin:0;font-family:sans-serif;padding:8px 4px 4px">
<p style="font-size:12px;color:#555;margin:0 0 6px">
  {len(rows)} findspots over {occupied} of the thirteen relations.
  <span style="display:inline-block;width:10px;height:10px;
    background:{ALLEN_COLOUR['finishedBy']};border-radius:2px;
    vertical-align:middle"></span>
  contemporary with or later than <em>{EVENT_LABEL}</em>
  ({qualifying}) &mdash;
  <span style="display:inline-block;width:10px;height:10px;background:#b4b2a9;
    border-radius:2px;vertical-align:middle"></span>
  wholly before it ({len(rows) - qualifying}).
</p>
<div style="position:relative;width:100%;height:{len(labels) * 30 + 60}px">
  <canvas id="allen"></canvas>
</div>
<script>
new Chart(document.getElementById("allen"), {{
  type: "bar",
  data: {{
    labels: {json.dumps(labels)},
    datasets: [{{
      data: {json.dumps(values)},
      backgroundColor: {json.dumps(colours)},
      borderWidth: 0, borderRadius: 3
    }}]
  }},
  options: {{
    indexAxis: "y", responsive: true, maintainAspectRatio: false,
    plugins: {{
      legend: {{display: false}},
      tooltip: {{callbacks: {{label: function (c) {{
        return " " + c.parsed.x + " findspot" + (c.parsed.x === 1 ? "" : "s");
      }}}}}}
    }},
    scales: {{
      x: {{beginAtZero: true, ticks: {{stepSize: 1, font: {{size: 12}}}},
          title: {{display: true, text: "Number of findspots",
                   font: {{size: 12}}}}}},
      y: {{ticks: {{font: {{size: 12, family: "monospace"}}}}}}
    }}
  }}
}});
</script>
</body></html>"""

Frame(chart, height=len(labels) * 30 + 130)

### The qualifying findspots on a timeline

The same rows, narrowed to the qualifying relations and drawn as intervals, with AD 9 marked. Sort by relation to separate the four groups: the 21 findspots ending in the year of the defeat, the ten beginning in AD 8 where the event's interval opens, the single one spanning it — Worms, 15 BC to AD 13 — and the six that begin only afterwards.

In [ ]:
rows = results['allen-relations-to-clades-variana']

# Figure: a Gantt-style timeline of every findspot still in use at, or first
# used after, the Clades Variana, with AD 9 marked. Ported from section 11 of
# notebook/clades_variana_temporal.ipynb.
#
# One deliberate change from the notebook version. There the colour map and the
# filter listed 'after', 'contains' and 'meets', which are not the relations
# this corpus actually contains: every bar that came out 'finishedBy' or
# 'startedBy' fell through to the grey default and could not be filtered for.
# Legend and filter are built from the data instead, so they cannot drift from
# it again.
#
# Runs in a Pyodide cell with `rows` (the query result) and the helpers from
# py/viz/_prelude.py in scope. It must end in a Frame.

# This figure and the one above share a query. The notebook does the same: one
# frame of all findspots with their relation, and the qualifying subset taken
# from it in Python rather than asked for a second time.
sites = [{"id": r["site"],
          "label": r["findspot"],
          "start": int(r["begin"]),
          "end": int(r["end"]),
          "rel": r["relation"]}
         for r in rows if r["relation"] in CONTEMPORARY_OR_LATER]

# Relations present, in Allen's order rather than in order of appearance.
present = [rel for rel in ALLEN_ORDER if any(s["rel"] == rel for s in sites)]

# Bar fill: the relation colour mixed into white. A tenth of the colour left
# the bars almost blank on screen and made the relations hard to tell apart;
# a fifth still lets the outline carry the meaning but is actually visible.
def _tint(hex_colour, strength=0.20):
    rgb = [int(hex_colour[k:k + 2], 16) for k in (1, 3, 5)]
    return "#%02x%02x%02x" % tuple(
        round(255 + (c - 255) * strength) for c in rgb)


fill = {rel: _tint(ALLEN_COLOUR[rel]) for rel in present}
stroke = {rel: ALLEN_COLOUR[rel] for rel in present}

span_min = min(s["start"] for s in sites)
span_max = max(s["end"] for s in sites)
pad = max(2, round((span_max - span_min) * 0.06))

legend = "".join(
    f'<span><b style="background:{ALLEN_COLOUR[rel]}"></b>{rel} '
    f'({sum(1 for s in sites if s["rel"] == rel)})</span>'
    for rel in present)
options = "".join(f'<option value="{rel}">{rel}</option>' for rel in present)

payload = json.dumps({"sites": sites, "fill": fill, "stroke": stroke,
                      "min": span_min - pad, "max": span_max + pad,
                      "event": EVENT_YEAR, "eventLabel": EVENT_LABEL,
                      "eventColour": EVENT_COLOUR})

style = """
  body{margin:0;font-family:sans-serif;padding:6px 4px 4px}
  #ctrl{display:flex;gap:12px;flex-wrap:wrap;align-items:center;
    padding:.3rem 0}
  #ctrl label{font-size:12px;color:#666}
  #ctrl select{font-size:12px;padding:2px 6px;border-radius:4px;
    border:1px solid #ccc;background:#fff;color:#333}
  #leg{display:flex;gap:12px;flex-wrap:wrap;font-size:11px;color:#555;
    padding:.2rem 0 .3rem}
  #leg span{display:flex;align-items:center;gap:4px}
  #leg b{width:10px;height:10px;border-radius:2px;display:inline-block}
"""

script = """
(function () {
  var C = JSON.parse(document.getElementById("payload").textContent);
  var se = document.getElementById("sort");
  var fe = document.getElementById("filter");
  var ce = document.getElementById("chart");

  function yr(v) { return v < 0 ? (-v) + " BC" : "AD " + v; }

  function ticks(lo, hi) {
    var step = (hi - lo) > 60 ? 10 : 5, out = [], t;
    for (t = Math.ceil(lo / step) * step; t <= hi; t += step) {
      // The event's own tick is always drawn, so a regular tick sitting on
      // top of it has to give way - AD 9 and AD 10 were printed over
      // each other.
      if (Math.abs(t - C.event) > 2) out.push(t);
    }
    out.push(C.event);
    return out.sort(function (a, b) { return a - b; });
  }

  function draw() {
    var s = se.value, f = fe.value;
    var d = f === "all" ? C.sites.slice()
                        : C.sites.filter(function (x) { return x.rel === f; });
    d.sort(function (a, b) {
      if (s === "start") return a.start - b.start || a.label.localeCompare(b.label);
      if (s === "end")   return a.end   - b.end   || a.label.localeCompare(b.label);
      if (s === "rel")   return a.rel.localeCompare(b.rel) || a.start - b.start;
      return a.label.localeCompare(b.label);
    });

    var LW = 200, RH = 21, RG = 4, MT = 38, MB = 36, CW = 480;
    var W = LW + CW + 24, H = MT + d.length * (RH + RG) + MB;
    function px(y) { return LW + (y - C.min) / (C.max - C.min) * CW; }

    var o = '<svg xmlns="http://www.w3.org/2000/svg" width="' + W + '"'
          + ' height="' + H + '" style="font-family:sans-serif;overflow:visible">';

    ticks(C.min, C.max).forEach(function (t) {
      var p = px(t), ev = (t === C.event);
      o += '<line x1="' + p + '" y1="' + (MT - 4) + '" x2="' + p + '"'
         + ' y2="' + (H - MB) + '" stroke="' + (ev ? C.eventColour : "#ddd") + '"'
         + ' stroke-width="' + (ev ? 1.5 : 0.5) + '"'
         + (ev ? ' stroke-dasharray="4 3"' : "") + "/>";
      o += '<text x="' + p + '" y="' + (MT - 8) + '" text-anchor="middle"'
         + ' font-size="10" fill="' + (ev ? C.eventColour : "#999") + '">'
         + yr(t) + "</text>";
    });

    d.forEach(function (x, i) {
      var y = MT + i * (RH + RG);
      var x1 = px(x.start), x2 = px(x.end), bw = Math.max(x2 - x1, 3);
      var sc = C.stroke[x.rel] || "#888", fc = C.fill[x.rel] || "#eee";
      o += '<text x="' + (LW - 6) + '" y="' + (y + RH * 0.72) + '"'
         + ' text-anchor="end" font-size="11" fill="#444">' + x.label + "</text>";
      o += '<rect x="' + x1 + '" y="' + (y + 2) + '" width="' + bw + '"'
         + ' height="' + (RH - 4) + '" rx="3" fill="' + fc + '"'
         + ' stroke="' + sc + '" stroke-width="1.2"><title>' + x.label
         + " \\u00b7 " + yr(x.start) + "\\u2013" + yr(x.end) + " \\u00b7 "
         + x.rel + "</title></rect>";
      var wide = bw > 70;
      o += '<text x="' + (wide ? x1 + bw / 2 : x2 + 4) + '"'
         + ' y="' + (y + RH * 0.72) + '"'
         + ' text-anchor="' + (wide ? "middle" : "start") + '" font-size="10"'
         + ' fill="' + (wide ? sc : "#999") + '">'
         + yr(x.start) + "\\u2013" + yr(x.end) + "</text>";
    });

    var ex = px(C.event);
    o += '<rect x="' + (ex - 44) + '" y="' + (H - MB + 4) + '" width="88"'
       + ' height="16" rx="3" fill="#faece7" stroke="' + C.eventColour + '"'
       + ' stroke-width=".8"/>';
    o += '<text x="' + ex + '" y="' + (H - MB + 15) + '" text-anchor="middle"'
       + ' font-size="10" fill="' + C.eventColour + '">' + C.eventLabel
       + "</text>";
    o += "</svg>";
    ce.innerHTML = o;
  }

  se.addEventListener("change", draw);
  fe.addEventListener("change", draw);
  draw();
})();
"""

timeline = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><style>{style}</style></head>
<body>
<div id="ctrl">
  <label for="sort">Sort by</label>
  <select id="sort">
    <option value="start">Start year</option>
    <option value="end">End year</option>
    <option value="rel">Allen relation</option>
    <option value="label">Findspot</option>
  </select>
  <label for="filter">Relation</label>
  <select id="filter">
    <option value="all">all ({len(sites)})</option>
    {options}
  </select>
</div>
<div id="leg">{legend}
  <span><b style="background:{EVENT_COLOUR};opacity:.7"></b>
    {EVENT_LABEL} {year(EVENT_YEAR)}</span>
</div>
<div id="chart"></div>
<script id="payload" type="application/json">{payload}</script>
<script>{script}</script>
</body></html>"""

Frame(timeline, height=38 + len(sites) * 25 + 36 + 110)

## 8 · How well is each horizon linked to gazetteers?

A practical linked-data question: can every findspot be reconciled against an external authority? Gaps mark the places where this graph cannot yet be joined to Wikidata or Pleiades.

In [ ]:
rows = show(g, r"""
SELECT ?horizon (COUNT(DISTINCT ?site) AS ?findspots)
                (COUNT(DISTINCT ?wikidata) AS ?withWikidata)
                (COUNT(DISTINCT ?pleiades) AS ?withPleiades)
WHERE {
  ?h a lado:ChronologicalHorizon ;
     skos:notation         ?horizon ;
     lado:hasHorizonMember ?site .
  OPTIONAL { ?site lado:wikidata ?wikidata }
  OPTIONAL { ?site lado:pleiades ?pleiades }
}
GROUP BY ?horizon
ORDER BY ?horizon""")

print(f"{len(rows)} rows")
table(rows)

## 9 · Where the findspots are

Every findspot carries a point geometry as a GeoSPARQL WKT literal, so the horizons can be read as a distribution as well as a sequence. WKT gives the coordinates in longitude-latitude order and prefixes the CRS IRI, hence the two REPLACE calls; the horizon and the authority links come along for the map's popups.

In [ ]:
rows = show(g, r"""
SELECT ?findspot ?horizon ?begin ?end ?latitude ?longitude
       ?wikidata ?pleiades
WHERE {
  ?site a fsl:Site ;
        rdfs:label                            ?findspot ;
        geosparql:hasGeometry/geosparql:asWKT ?wkt ;
        time:hasBeginning/time:inXSDgYear     ?beginYear ;
        time:hasEnd/time:inXSDgYear           ?endYear .

  OPTIONAL {
    ?h a lado:ChronologicalHorizon ;
       skos:notation         ?horizon ;
       lado:hasHorizonMember ?site .
  }
  OPTIONAL { ?site lado:wikidata ?wikidata }
  OPTIONAL { ?site lado:pleiades ?pleiades }

  BIND(xsd:integer(STR(?beginYear)) AS ?begin)
  BIND(xsd:integer(STR(?endYear))   AS ?end)
  BIND(REPLACE(STR(?wkt),
       "^.*POINT\\s*\\(\\s*(\\S+)\\s+(\\S+)\\s*\\).*$", "$1") AS ?longitude)
  BIND(REPLACE(STR(?wkt),
       "^.*POINT\\s*\\(\\s*(\\S+)\\s+(\\S+)\\s*\\).*$", "$2") AS ?latitude)
}
ORDER BY ?horizon ?findspot""")
results['findspot-coordinates'] = rows

print(f"{len(rows)} rows")
table(rows)

### The findspots on a map

Points are coloured by horizon, cold to warm, and each horizon is its own layer — switch them off in the control at the top right to see one phase at a time. The Rhine, the Danube and the roads between them account for all but one findspot: Conimbriga, in horizon 3, sits 1411 km from its nearest neighbour in the corpus. It is worth switching horizon 3 on and off, because that single point is what makes the horizon's convex hull so much larger than the others — a reminder that a wide hull can mean one distant site rather than a wide distribution.

In [ ]:
rows = results['findspot-coordinates']

# Figure: the findspots on a slippy map, coloured and grouped by chronological
# horizon. There is no counterpart in notebook/clades_variana_temporal.ipynb —
# this is the spatial reading of the same graph, added because every findspot
# carries a geosparql:asWKT point and the horizons are as much a distribution
# as a sequence.
#
# The tiles come from openstreetmap.org, so this is the one figure in the
# notebook that makes a network request beyond the graph itself. Everything
# else, including which point sits where, is computed from the Turtle file.
#
# Runs in a Pyodide cell with `rows` (the query result) and the helpers from
# py/viz/_prelude.py in scope. It must end in a Frame.

# One entry per findspot. The OPTIONAL blocks in the query would multiply rows
# if a findspot ever gained a second authority link, so collapse on the label.
sites = {}
for r in rows:
    sites[r["findspot"]] = {
        "label": r["findspot"],
        "horizon": r["horizon"],
        "start": int(r["begin"]),
        "end": int(r["end"]),
        "lat": float(r["latitude"]),
        "lon": float(r["longitude"]),
        "span": f'{year(r["begin"])}\u2013{year(r["end"])}',
        "wikidata": r.get("wikidata"),
        "pleiades": r.get("pleiades"),
    }

horizons = sorted({s["horizon"] for s in sites.values()})
by_horizon = {h: sum(1 for s in sites.values() if s["horizon"] == h)
              for h in horizons}

payload = json.dumps({
    "sites": sorted(sites.values(), key=lambda s: (s["horizon"], s["label"])),
    "colour": {h: HORIZON_COLOUR.get(h, "#666666") for h in horizons},
})

legend = "".join(
    f'<span><b style="background:{HORIZON_COLOUR.get(h, "#666")}"></b>'
    f'Horizon {escape(h)} ({by_horizon[h]})</span>' for h in horizons)

# Leaflet sizes the map from its container, and a container given a percentage
# height inside an unsized body collapses to nothing. So the frame height is
# fixed here and the map gets what is left after the legend.
FRAME_HEIGHT = 580
MAP_HEIGHT = FRAME_HEIGHT - 46

style = f"""
  body{{margin:0;font-family:sans-serif;padding:6px 4px 4px}}
  #leg{{display:flex;gap:12px;flex-wrap:wrap;font-size:11px;color:#555;
    padding:.2rem 0 .4rem}}
  #leg span{{display:flex;align-items:center;gap:4px}}
  #leg b{{width:10px;height:10px;border-radius:50%;display:inline-block}}
  #map{{width:100%;height:{MAP_HEIGHT}px;border:1px solid #ddd;
    border-radius:4px}}
  .leaflet-popup-content{{font-size:12px;line-height:1.45;margin:8px 12px}}
  .leaflet-popup-content b{{font-size:13px}}
  .leaflet-popup-content .meta{{color:#666}}
  .leaflet-popup-content a{{color:#185fa5}}
"""

script = """
(function () {
  var C = JSON.parse(document.getElementById("payload").textContent);
  var map = L.map("map", {scrollWheelZoom: false});

  L.tileLayer("https://tile.openstreetmap.org/{z}/{x}/{y}.png", {
    maxZoom: 12, minZoom: 3,
    attribution: '&copy; <a href="https://www.openstreetmap.org/copyright"'
               + ' target="_blank" rel="noopener">OpenStreetMap</a> contributors'
  }).addTo(map);

  function link(url, text) {
    if (!url) return "";
    return ' <a href="' + url + '" target="_blank" rel="noopener">'
         + text + "</a>";
  }

  var layers = {}, bounds = [];
  C.sites.forEach(function (s) {
    var colour = C.colour[s.horizon] || "#666666";
    var marker = L.circleMarker([s.lat, s.lon], {
      radius: 6, color: "#ffffff", weight: 1.5,
      fillColor: colour, fillOpacity: 0.9
    });
    var authorities = link(s.wikidata, "Wikidata") + link(s.pleiades, "Pleiades");
    marker.bindPopup(
      "<b>" + s.label + "</b><br>"
      + '<span class="meta">' + s.span + " \\u00b7 horizon " + s.horizon
      + "</span>" + (authorities ? "<br>" + authorities : ""));
    marker.bindTooltip(s.label, {direction: "top", offset: [0, -6]});

    var key = "Horizon " + s.horizon;
    if (!layers[key]) layers[key] = L.layerGroup().addTo(map);
    marker.addTo(layers[key]);
    bounds.push([s.lat, s.lon]);
  });

  L.control.layers(null, layers, {collapsed: false}).addTo(map);
  map.fitBounds(bounds, {padding: [24, 24]});
})();
"""

site_map = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<link rel="stylesheet"
      href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.css">
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.js">
</script>
<style>{style}</style>
</head>
<body>
<div id="leg">{legend}</div>
<div id="map"></div>
<script id="payload" type="application/json">{payload}</script>
<script>{script}</script>
</body></html>"""

Frame(site_map, height=FRAME_HEIGHT)

## 10 · What each assemblage is made of

The first query over the evidence rather than over the result. The counts live in a second graph, built by `py/services_to_rdf.py` from the workbook and merged with this one on the fly — the notice above says when it is being fetched. Percentages are computed here rather than stored: a derived number kept beside the number it derives from is a second thing that has to stay true.

In [ ]:
# This query also needs the service-type counts, so it
# runs against the two graphs merged.
rows = show(g + services, r"""
SELECT ?findspot ?begin ?end ?horizon ?slug ?type ?rank ?sherds ?total
WHERE {
  ?obs aeont:atFindspot  ?site ;
       aeont:serviceType ?concept ;
       aeont:sherdCount  ?sherds .

  ?site rdfs:label                        ?findspot ;
        time:hasBeginning/time:inXSDgYear ?beginYear ;
        time:hasEnd/time:inXSDgYear       ?endYear .

  # The label is what the figure prints, the slug is what it keys its
  # colours by. Keeping the two apart means a figure does not quietly
  # lose its palette the moment the display language changes.
  ?concept skos:prefLabel        ?type ;
           aeont:typologicalRank ?rank .
  FILTER(lang(?type) = "en")
  BIND(REPLACE(STR(?concept), "^.*/", "") AS ?slug)

  OPTIONAL {
    ?h a lado:ChronologicalHorizon ;
       skos:notation         ?horizon ;
       lado:hasHorizonMember ?site .
  }

  # The findspot's own total, so the share can be worked out per row.
  {
    SELECT ?site (SUM(?k) AS ?total)
    WHERE { ?o aeont:atFindspot ?site ; aeont:sherdCount ?k }
    GROUP BY ?site
  }

  BIND(xsd:integer(STR(?beginYear)) AS ?begin)
  BIND(xsd:integer(STR(?endYear))   AS ?end)
}
ORDER BY ?begin ?findspot ?rank""")
results['service-composition'] = rows

print(f"{len(rows)} rows")
table(rows)

### Service composition per findspot

The same figure as `output/events_timeline_by_service_en.svg`: each bar sits on the time axis over the findspot's interval and is split into the shares of its service types, with the horizons separated. The shift from the red Service I types to the green Service II ones is plain, and so is what the printed version shows about horizon 5 — six findspots, far to the right, almost pure Service II. Switch to equal-width bars to compare compositions without the intervals getting in the way.

In [ ]:
rows = results['service-composition']

# Figure: each findspot's assemblage split into its service types, drawn on the
# time axis over the findspot's interval and grouped by horizon. The browser
# counterpart of output/events_timeline_by_service_en.svg, and drawn to match
# it: latest horizon at the top, dashed rules between horizons, horizon labels
# down the right, a leader line from the name to the bar, and the same palette.
#
# The bar *positions* carry the chronology, the *segments* carry the
# composition. Because a long interval then gets a wide bar, a second mode
# gives every findspot the same width, which is the only way to compare
# compositions of findspots whose intervals differ by a factor of five. The
# printed figure has to choose; this one does not.
#
# Percentages are not stored in the graph. The query computes them from
# aeont:sherdCount, so this figure and the printed one cannot drift apart:
# there is one copy of the counts and no copy of the shares.
#
# Runs in a Pyodide cell with `rows` (the query result) and the helpers from
# py/viz/_prelude.py in scope. It must end in a Frame.

sites, order = {}, []
for r in rows:
    name = r["findspot"]
    if name not in sites:
        sites[name] = {"label": name,
                       "begin": int(r["begin"]), "end": int(r["end"]),
                       "horizon": r.get("horizon") or "\u2013",
                       "total": int(r["total"]), "parts": []}
        order.append(name)
    sites[name]["parts"].append({
        "slug": r["slug"],
        "type": r["type"],
        "sherds": int(r["sherds"]),
        "share": 100 * int(r["sherds"]) / int(r["total"]),
    })

# Colours are keyed by the concept's slug, not by its label: the label is
# display text and changes with the language, the slug is the identifier. An
# unmatched key used to leave every bar grey and the legend empty, which reads
# as a design choice rather than a fault, so it is an error now.
missing = ({p["slug"] for s in sites.values() for p in s["parts"]}
           - set(SERVICE_COLOUR))
assert not missing, (
    f"no colour for {sorted(missing)} \u2014 SERVICE_COLOUR in "
    f"py/viz/_prelude.py is keyed by concept slug")

label_of = {p["slug"]: p["type"] for s in sites.values() for p in s["parts"]}
present = [slug for slug in SERVICE_COLOUR if slug in label_of]

legend = "".join(
    f'<span><b style="background:{SERVICE_COLOUR[slug]}"></b>'
    f'{escape(label_of[slug])}</span>' for slug in present)

payload = json.dumps({
    "sites": [sites[n] for n in order],
    "colour": {slug: SERVICE_COLOUR[slug] for slug in present},
    "min": min(s["begin"] for s in sites.values()) - 2,
    "max": max(s["end"] for s in sites.values()) + 1,
})

style = """
  body{margin:0;font-family:sans-serif;padding:6px 4px 4px}
  #ctrl{display:flex;gap:12px;flex-wrap:wrap;align-items:center;padding:.2rem 0}
  #ctrl label{font-size:12px;color:#666}
  #ctrl select{font-size:12px;padding:2px 6px;border-radius:4px;
    border:1px solid #ccc;background:#fff;color:#333}
  #leg{display:flex;gap:10px;flex-wrap:wrap;font-size:11px;color:#555;
    padding:.2rem 0 .4rem}
  #leg span{display:flex;align-items:center;gap:4px}
  #leg b{width:10px;height:10px;border-radius:2px;display:inline-block}
  #chart{overflow-x:auto}
"""

script = """
(function () {
  var C = JSON.parse(document.getElementById("payload").textContent);
  var me = document.getElementById("mode");
  var ce = document.getElementById("chart");

  function yr(v) { return v < 0 ? (-v) + " BC" : "AD " + v; }

  function draw() {
    var onAxis = me.value === "axis";

    // Latest horizon at the top, and within a horizon the latest start first,
    // which is how the printed figure orders its rows.
    var d = C.sites.slice().sort(function (a, b) {
      if (a.horizon !== b.horizon) return a.horizon < b.horizon ? 1 : -1;
      return b.begin - a.begin || a.label.localeCompare(b.label);
    });

    var LW = 232, RH = 15, RG = 7, MT = 14, MB = 40, CW = 640, RW = 74;
    var W = LW + CW + RW, H = MT + d.length * (RH + RG) + MB;
    function px(y) { return LW + (y - C.min) / (C.max - C.min) * CW; }

    var o = '<svg xmlns="http://www.w3.org/2000/svg" width="' + W + '"'
          + ' height="' + H + '" style="font-family:sans-serif">';

    if (onAxis) {
      var step = 5, t;
      for (t = Math.ceil(C.min / step) * step; t <= C.max; t += step) {
        var p = px(t);
        o += '<line x1="' + p + '" y1="' + MT + '" x2="' + p + '"'
           + ' y2="' + (H - MB) + '" stroke="#f0f0f0" stroke-width="1"/>';
        o += '<text x="' + p + '" y="' + (H - MB + 14) + '" font-size="9.5"'
           + ' fill="#777" text-anchor="end" transform="rotate(-35 ' + p + ','
           + (H - MB + 14) + ')">' + yr(t) + "</text>";
      }
    }

    var prev = null, blockTop = MT;
    d.forEach(function (x, i) {
      var y = MT + i * (RH + RG);

      if (prev !== null && x.horizon !== prev) {
        var ry = y - RG / 2;
        o += '<line x1="4" y1="' + ry + '" x2="' + (LW + CW) + '"'
           + ' y2="' + ry + '" stroke="#bbb" stroke-width="1"'
           + ' stroke-dasharray="5 4"/>';
        o += horizonLabel(prev, blockTop, ry);
        blockTop = ry;
      }
      prev = x.horizon;

      var x1 = onAxis ? px(x.begin) : LW;
      var w = onAxis ? Math.max(px(x.end) - x1, 2.5) : CW;

      o += '<text x="' + (LW - 10) + '" y="' + (y + RH * 0.8) + '"'
         + ' text-anchor="end" font-size="10.5" font-weight="600"'
         + ' fill="#333">' + x.label + "</text>";
      if (onAxis && x1 > LW + 2) {
        // Faint enough not to compete with the bars, dark enough to survive
        // being looked at on a screen - #d5d5d5 at 0.8 was invisible.
        o += '<line x1="' + (LW - 5) + '" y1="' + (y + RH / 2) + '"'
           + ' x2="' + (x1 - 2) + '" y2="' + (y + RH / 2) + '"'
           + ' stroke="#9aa0a6" stroke-width="1" stroke-dasharray="1 3"/>';
      }

      var cursor = x1;
      x.parts.forEach(function (p) {
        var seg = p.share / 100 * w;
        o += '<rect x="' + cursor + '" y="' + y + '"'
           + ' width="' + Math.max(seg, 0.35) + '" height="' + RH + '"'
           + ' fill="' + (C.colour[p.slug] || "#ccc") + '">'
           + "<title>" + x.label + " (" + yr(x.begin) + "\\u2013" + yr(x.end)
           + ") \\u2014 " + p.type + ": " + p.sherds + " sherds, "
           + p.share.toFixed(1) + "%</title></rect>";
        cursor += seg;
      });

      if (!onAxis) {
        o += '<text x="' + (LW + CW + 8) + '" y="' + (y + RH * 0.8) + '"'
           + ' font-size="9.5" fill="#999">' + x.total + "</text>";
      }
    });

    o += horizonLabel(prev, blockTop, MT + d.length * (RH + RG) - RG / 2);
    o += "</svg>";
    ce.innerHTML = o;

    function horizonLabel(h, top, bottom) {
      var mid = (top + bottom) / 2, x = LW + CW + (onAxis ? 22 : 40);
      var s = '<line x1="' + (x - 8) + '" y1="' + (top + 3) + '" x2="'
            + (x - 8) + '" y2="' + (bottom - 3) + '" stroke="#999"'
            + ' stroke-width="1"/>';
      s += '<text x="' + x + '" y="' + mid + '" font-size="11"'
         + ' font-weight="600" fill="#444" text-anchor="middle"'
         + ' transform="rotate(-90 ' + x + ',' + mid + ')">Horizon ' + h
         + "</text>";
      return s;
    }
  }

  me.addEventListener("change", draw);
  draw();
})();
"""

composition = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><style>{style}</style></head>
<body>
<div id="ctrl">
  <label for="mode">Bars</label>
  <select id="mode">
    <option value="axis">On the time axis</option>
    <option value="equal">Equal width (composition only)</option>
  </select>
  <span style="font-size:11px;color:#888">
    {len(sites)} findspots &middot; segment = that type's share of the
    findspot's sherds &middot; hover for counts
  </span>
</div>
<div id="leg">{legend}</div>
<div id="chart"></div>
<script id="payload" type="application/json">{payload}</script>
<script>{script}</script>
</body></html>"""

Frame(composition, height=14 + len(sites) * 22 + 40 + 120)

## 11 · How tightly each horizon is defined

The RGZM variance and quality of Allard Mees, computed in the query as far as SPARQL reaches. Every sherd counts as one observation valued by the rank of its sub-type within its group, so N, the sum of the ranks and the sum of their squares are enough to derive the rest — which is fortunate, because rdflib's SPARQL has neither SQRT nor EXP and `s` and `q = exp(−CV)` have to be taken in Python. Both rank readings come back from the one query: the stage reading this project reports, and the column reading kept for comparison. `COUNT(DISTINCT ?rank)` reports how many ranks a group actually holds, so a cell whose value follows from the rank assignment alone can be told apart from one that was measured.

In [ ]:
# This query also needs the service-type counts, so it
# runs against the two graphs merged.
rows = show(g + services, r"""
SELECT ?horizon ?group (SUM(?sherds) AS ?sherds)
       (SUM(?stage * ?sherds)            AS ?sumStage)
       (SUM(?stage * ?stage * ?sherds)   AS ?sumStageSq)
       (COUNT(DISTINCT ?stage)           AS ?stageSteps)
       (SUM(?column * ?sherds)           AS ?sumColumn)
       (SUM(?column * ?column * ?sherds) AS ?sumColumnSq)
       (COUNT(DISTINCT ?column)          AS ?columnSteps)
WHERE {
  ?h a lado:ChronologicalHorizon ;
     skos:notation         ?horizon ;
     lado:hasHorizonMember ?site .

  ?obs aeont:atFindspot  ?site ;
       aeont:serviceType ?concept ;
       aeont:sherdCount  ?sherds .

  # The group is the concept's top concept, reached through skos:broader.
  ?concept aeont:stageRank  ?stage ;
           aeont:columnRank ?column ;
           skos:broader*    ?top .
  ?top skos:topConceptOf ?scheme ;
       skos:prefLabel    ?group .
  FILTER(lang(?group) = "en")
}
GROUP BY ?horizon ?group
ORDER BY ?horizon ?group""")
results['service-within-group-variability'] = rows

print(f"{len(rows)} rows")
table(rows)

### Within-group variance and quality per horizon

Service I is the group to read, and it says the same thing under either reading: quality rises steadily from the earliest horizon to the latest, and the two readings order the horizons identically, so the finding does not rest on the choice between them. What the switch does change is Service II — grey under the stage reading, because it holds a single stage and its numbers follow from the rank assignment rather than from the sherds, and measured under the sub-type reading, where cup and plate count as two steps. Whether that second reading says anything archaeological is exactly the question the stage reading answers with no.

In [ ]:
rows = results['service-within-group-variability']

# Figure: the RGZM within-group variance and quality per horizon and service
# group. The browser counterpart of output/service_group_variability_en.svg and
# drawn to match it: two panels side by side, variance on RdYlGn_r from 0 to the
# largest value, quality on RdYlGn from 0 to 1, latest horizon at the top, a
# colourbar under each panel, and grey for the cells whose value follows from
# the rank assignment rather than from the sherds.
#
# One thing the printed figure cannot do is offered here instead of a second
# page: the rank reading can be switched. The stage reading is what the project
# reports — a cup and a plate of one stage share a rank, because a change of
# vessel form is not a chronological step. The sub-type reading counts every
# type as a step and is kept so the alternative can be re-checked; it orders the
# horizons the same way.
#
# The query returns N, Σ(rank·count) and Σ(rank²·count); the standard deviation
# and exp(−CV) are taken by rgzm() in py/viz/_prelude.py, because rdflib's
# SPARQL has neither SQRT nor EXP.
#
# Runs in a Pyodide cell with `rows` (the query result) and the helpers from
# py/viz/_prelude.py in scope. It must end in a Frame.

READINGS = [
    ("stage", "ranks by stage only", "stageSteps", "sumStage", "sumStageSq"),
    ("column", "ranks by sub-type", "columnSteps", "sumColumn", "sumColumnSq"),
]

horizons, groups, data = [], [], {}
for r in rows:
    horizon, group, n = r["horizon"], r["group"], int(r["sherds"])
    if horizon not in horizons:
        horizons.append(horizon)
    if group not in groups:
        groups.append(group)
    for key, _note, steps_field, sum_field, sumsq_field in READINGS:
        s, q = rgzm(n, float(r[sum_field]), float(r[sumsq_field]))
        data[(key, horizon, group)] = {
            "n": n, "s": s, "q": q, "measured": int(r[steps_field]) > 1}

horizons.sort()

payload = json.dumps({
    "readings": {k: note for k, note, *_ in READINGS},
    "horizons": horizons,
    "groups": groups,
    # Keyed "reading|horizon|group", so switching the reading is a redraw
    # rather than a second query.
    "cells": {f"{k}|{h}|{g}": v for (k, h, g), v in data.items()},
    "vmax": {k: max((v["s"] for (rk, _h, _g), v in data.items()
                     if rk == k and v["measured"]), default=1.0)
             for k, *_ in READINGS},
    "rdylgn": RDYLGN,
    "grey": STRUCTURAL_GREY,
})

# The variance panel is scaled to its own largest value, so its bar is labelled
# with that; the quality panel always runs 0 to 1, as in the printed figure.
bars = {k: colourbar(RDYLGN[::-1], 0, v, width=250, fmt="{:.2f}")
        for k, v in json.loads(payload)["vmax"].items()}
bar_qual = colourbar(RDYLGN, 0, 1, width=250, fmt="{:.1f}")

style = """
  body{margin:0;font-family:sans-serif;padding:6px 4px 4px;color:#333}
  #ctrl{display:flex;gap:10px;align-items:center;font-size:12px;color:#666;
    padding:0 0 .6rem}
  #ctrl select{font-size:12px;padding:2px 6px;border-radius:4px;
    border:1px solid #ccc;background:#fff;color:#333}
  #ctrl em{font-style:normal;color:#999}
  .panels{display:flex;gap:26px;flex-wrap:wrap;align-items:flex-start}
  .panel h3{margin:0 0 8px;font-size:12.5px;font-weight:700;text-align:center}
  table{border-collapse:separate;border-spacing:2px}
  th{font-size:11.5px;font-weight:700;padding:0 4px 4px;text-align:center}
  th.side{text-align:right;white-space:nowrap}
  td{width:104px;height:56px;text-align:center;line-height:1.25;cursor:help}
  .v{display:block;font-size:15px;font-weight:600}
  .n{display:block;font-size:9.5px;opacity:.65}
  .cbar{margin-top:8px;text-align:center}
  .cbar span{display:block;font-size:10px;color:#999;margin-bottom:2px}
  .foot{font-size:11px;color:#888;max-width:82ch;line-height:1.55;
    padding:.8rem 0 0}
  .foot b{display:inline-block;width:11px;height:11px;background:#eeeeee;
    border:1px solid #ddd;vertical-align:middle}
"""

script = """
(function () {
  var C = JSON.parse(document.getElementById("payload").textContent);
  var se = document.getElementById("reading");
  var ne = document.getElementById("note");

  // The group colours of the printed figure's header row.
  var HEAD = {"Oblique-rim plate": "#1f77b4", "Service I": "#d62728",
              "Service II": "#2ca02c"};

  function ramp(stops, t) {
    t = Math.min(Math.max(t, 0), 1);
    var span = t * (stops.length - 1);
    var i = Math.min(Math.floor(span), stops.length - 2), f = span - i;
    var out = "#", k, v;
    for (k = 1; k < 6; k += 2) {
      v = Math.round(parseInt(stops[i].substr(k, 2), 16)
        + (parseInt(stops[i + 1].substr(k, 2), 16)
           - parseInt(stops[i].substr(k, 2), 16)) * f);
      out += ("0" + v.toString(16)).slice(-2);
    }
    return out;
  }

  function ink(hex) {
    var r = parseInt(hex.substr(1, 2), 16), g = parseInt(hex.substr(3, 2), 16),
        b = parseInt(hex.substr(5, 2), 16);
    return (0.299 * r + 0.587 * g + 0.114 * b) > 150 ? "#1a1a1a" : "#ffffff";
  }

  function panel(kind, reading) {
    var vmax = kind === "s" ? C.vmax[reading] : 1;
    var stops = kind === "s" ? C.rdylgn.slice().reverse() : C.rdylgn;
    var h = "<table><thead><tr><th></th>";
    C.groups.forEach(function (g) {
      h += '<th style="color:' + (HEAD[g] || "#444") + '">' + g + "</th>";
    });
    h += "</tr></thead><tbody>";
    C.horizons.slice().reverse().forEach(function (hz) {
      h += '<tr><th class="side">Horizon ' + hz + "</th>";
      C.groups.forEach(function (g) {
        var c = C.cells[reading + "|" + hz + "|" + g];
        if (!c) {
          h += '<td style="background:' + C.grey + ';color:#bbb">&mdash;</td>';
          return;
        }
        var v = kind === "s" ? c.s : c.q;
        var fill = c.measured ? ramp(stops, vmax ? v / vmax : 0) : C.grey;
        var tone = c.measured ? ink(fill) : "#888";
        var tip = g + ", horizon " + hz + ": " + c.n + " sherds, s = "
                + c.s.toFixed(3) + ", q = " + c.q.toFixed(3)
                + (c.measured ? "" : " \\u2014 one rank only, so this follows"
                   + " from the rank assignment rather than from the sherds");
        h += '<td style="background:' + fill + ';color:' + tone + '"'
           + ' title="' + tip + '"><span class="v">' + v.toFixed(2)
           + '</span><span class="n">n=' + c.n + "</span></td>";
      });
      h += "</tr>";
    });
    return h + "</tbody></table>";
  }

  function draw() {
    var reading = se.value;
    ne.textContent = C.readings[reading];
    document.getElementById("pv").innerHTML = panel("s", reading);
    document.getElementById("pq").innerHTML = panel("q", reading);
    Array.prototype.forEach.call(
      document.querySelectorAll("[data-bar]"), function (el) {
        el.style.display = el.getAttribute("data-bar") === reading ? "" : "none";
      });
  }

  se.addEventListener("change", draw);
  draw();
})();
"""

var_bars = "".join(f'<div data-bar="{k}">{bar}</div>' for k, bar in bars.items())

heatmap = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><style>{style}</style></head>
<body>
<div id="ctrl">
  <label for="reading">Rank reading</label>
  <select id="reading">
    <option value="stage">Stage &mdash; reported</option>
    <option value="column">Sub-type &mdash; for comparison</option>
  </select>
  <em id="note"></em>
</div>
<div class="panels">
  <div class="panel">
    <h3>Within-group variance &nbsp;(STDDEV_SAMP of sub-type ranks)</h3>
    <div id="pv"></div>
    <div class="cbar"><span>variance</span>{var_bars}</div>
  </div>
  <div class="panel">
    <h3>Within-group quality &nbsp;(q = exp(&minus;CV))</h3>
    <div id="pq"></div>
    <div class="cbar"><span>quality</span>{bar_qual}</div>
  </div>
</div>
<p class="foot">
  Every sherd is one observation valued by the rank of its sub-type within its
  group. <b></b> grey: the group holds a single rank, so <em>s</em> = 0 and
  <em>q</em> = 1 follow from the rank assignment rather than from the material.
  Hover a cell for both figures and the sherd count.
</p>
<script id="payload" type="application/json">{payload}</script>
<script>{script}</script>
</body></html>"""

Frame(heatmap, height=len(horizons) * 62 + 250)

## Explore

The graph is in `base` and stays in scope, as does `results`, which holds
the rows of every query above under its id. A few things worth trying:

- Change the year in the *Clades Variana* queries to 15 BC, the start of
  the Drusus campaigns: `ae:event_Drusus_campaigns` is in the graph too,
  with its own interval.
- Swap `lado:hasHorizonMember` for `geosparql:memberOf` in the other
  direction — the graph carries both, and the second also reaches the
  period clusters and the corpus collection.
- Ask for the findspots a horizon's convex hull covers, via
  `geosparql:hasGeometry`, and compare it with the points on the map.
- List what the graph is actually made of: that is the cell below.

In [ ]:
rows = show(g, r"""
SELECT ?class (COUNT(?s) AS ?instances)
WHERE {
  ?s a ?c .
  BIND(REPLACE(REPLACE(STR(?c), "^.*#", ""), "^.*/", "") AS ?class)
}
GROUP BY ?class
ORDER BY DESC(?instances) ?class""")
table(rows)

---

Produced for the CAA 2026 contribution *Seriation, Chronology and Linked
Open Data* (LEIZA). The graph is built by `py/alligator_to_clean_rdf.py`
from the output of [Alligator](https://tools.leiza.de/alligator/);
vocabularies: CIDOC-CRM, OWL-Time, GeoSPARQL, SKOS and
[LADO](http://archaeology.link/). Notebook structure follows the
NFDI4Objects OER template.